In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from privacy_estimates.experiments.aml import JobList, Job
from sklearn.metrics import roc_curve, auc
from tempfile import TemporaryDirectory
from datasets import load_from_disk
from latex import Project
from dotenv import load_dotenv

In [ ]:
jobs = JobList.from_urls([
    # prefix_length = 0
    "https://ml.azure.com/experiments/id/cbd45cd3-4fd8-4922-b82a-124527cc98ee/runs/mighty_bone_29pr16nz24?wsid=/subscriptions/acc09744-1ee3-4242-b375-93421c63af0c/resourceGroups/PPML/providers/Microsoft.MachineLearningServices/workspaces/M365Research-PPML-EUS&tid=72f988bf-86f1-41af-91ab-2d7cd011db47",
    "https://ml.azure.com/experiments/id/cbd45cd3-4fd8-4922-b82a-124527cc98ee/runs/bright_oxygen_k8mzymr8p5?wsid=/subscriptions/acc09744-1ee3-4242-b375-93421c63af0c/resourceGroups/PPML/providers/Microsoft.MachineLearningServices/workspaces/M365Research-PPML-EUS&tid=72f988bf-86f1-41af-91ab-2d7cd011db47",
    "https://ml.azure.com/experiments/id/cbd45cd3-4fd8-4922-b82a-124527cc98ee/runs/9b7ec13e-7d29-4e98-88e4-18a6f4b2f0b9?wsid=/subscriptions/acc09744-1ee3-4242-b375-93421c63af0c/resourceGroups/PPML/providers/Microsoft.MachineLearningServices/workspaces/M365Research-PPML-EUS&tid=72f988bf-86f1-41af-91ab-2d7cd011db47",
    "https://ml.azure.com/experiments/id/cbd45cd3-4fd8-4922-b82a-124527cc98ee/runs/blue_pin_5v9pq3wx2z?wsid=/subscriptions/acc09744-1ee3-4242-b375-93421c63af0c/resourceGroups/PPML/providers/Microsoft.MachineLearningServices/workspaces/M365Research-PPML-EUS&tid=72f988bf-86f1-41af-91ab-2d7cd011db47",
    "https://ml.azure.com/experiments/id/cbd45cd3-4fd8-4922-b82a-124527cc98ee/runs/magenta_muscle_23qs57ykv7?wsid=/subscriptions/acc09744-1ee3-4242-b375-93421c63af0c/resourceGroups/PPML/providers/Microsoft.MachineLearningServices/workspaces/M365Research-PPML-EUS&tid=72f988bf-86f1-41af-91ab-2d7cd011db47",
    "https://ml.azure.com/experiments/id/cbd45cd3-4fd8-4922-b82a-124527cc98ee/runs/busy_rainbow_p2zzs5pyc9?wsid=/subscriptions/acc09744-1ee3-4242-b375-93421c63af0c/resourceGroups/PPML/providers/Microsoft.MachineLearningServices/workspaces/M365Research-PPML-EUS&tid=72f988bf-86f1-41af-91ab-2d7cd011db47",
    "https://ml.azure.com/experiments/id/cbd45cd3-4fd8-4922-b82a-124527cc98ee/runs/nifty_puppy_q6z18gy093?wsid=/subscriptions/acc09744-1ee3-4242-b375-93421c63af0c/resourceGroups/PPML/providers/Microsoft.MachineLearningServices/workspaces/M365Research-PPML-EUS&tid=72f988bf-86f1-41af-91ab-2d7cd011db47",
    "https://ml.azure.com/experiments/id/cbd45cd3-4fd8-4922-b82a-124527cc98ee/runs/frank_calypso_b3yfnysx5k?wsid=/subscriptions/acc09744-1ee3-4242-b375-93421c63af0c/resourceGroups/PPML/providers/Microsoft.MachineLearningServices/workspaces/M365Research-PPML-EUS&tid=72f988bf-86f1-41af-91ab-2d7cd011db47",
    "https://ml.azure.com/experiments/id/cbd45cd3-4fd8-4922-b82a-124527cc98ee/runs/happy_scooter_71092vwkkx?wsid=/subscriptions/acc09744-1ee3-4242-b375-93421c63af0c/resourceGroups/PPML/providers/Microsoft.MachineLearningServices/workspaces/M365Research-PPML-EUS&tid=72f988bf-86f1-41af-91ab-2d7cd011db47",
    # prefix_length = 10
    "https://ml.azure.com/experiments/id/cbd45cd3-4fd8-4922-b82a-124527cc98ee/runs/lemon_goat_31mdlk97vc?wsid=/subscriptions/acc09744-1ee3-4242-b375-93421c63af0c/resourceGroups/PPML/providers/Microsoft.MachineLearningServices/workspaces/M365Research-PPML-EUS&tid=72f988bf-86f1-41af-91ab-2d7cd011db47",
    "https://ml.azure.com/experiments/id/cbd45cd3-4fd8-4922-b82a-124527cc98ee/runs/upbeat_rose_g9dxfq15nd?wsid=/subscriptions/acc09744-1ee3-4242-b375-93421c63af0c/resourceGroups/PPML/providers/Microsoft.MachineLearningServices/workspaces/M365Research-PPML-EUS&tid=72f988bf-86f1-41af-91ab-2d7cd011db47",
    "https://ml.azure.com/experiments/id/cbd45cd3-4fd8-4922-b82a-124527cc98ee/runs/strong_worm_hcb1th9ks8?wsid=/subscriptions/acc09744-1ee3-4242-b375-93421c63af0c/resourceGroups/PPML/providers/Microsoft.MachineLearningServices/workspaces/M365Research-PPML-EUS&tid=72f988bf-86f1-41af-91ab-2d7cd011db47",
    "https://ml.azure.com/experiments/id/cbd45cd3-4fd8-4922-b82a-124527cc98ee/runs/maroon_juice_93swc916bt?wsid=/subscriptions/acc09744-1ee3-4242-b375-93421c63af0c/resourceGroups/PPML/providers/Microsoft.MachineLearningServices/workspaces/M365Research-PPML-EUS&tid=72f988bf-86f1-41af-91ab-2d7cd011db47",
    "https://ml.azure.com/experiments/id/cbd45cd3-4fd8-4922-b82a-124527cc98ee/runs/icy_street_qky03dhvnh?wsid=/subscriptions/acc09744-1ee3-4242-b375-93421c63af0c/resourceGroups/PPML/providers/Microsoft.MachineLearningServices/workspaces/M365Research-PPML-EUS&tid=72f988bf-86f1-41af-91ab-2d7cd011db47",
    "https://ml.azure.com/experiments/id/cbd45cd3-4fd8-4922-b82a-124527cc98ee/runs/gentle_bear_f39thbwf3f?wsid=/subscriptions/acc09744-1ee3-4242-b375-93421c63af0c/resourceGroups/PPML/providers/Microsoft.MachineLearningServices/workspaces/M365Research-PPML-EUS&tid=72f988bf-86f1-41af-91ab-2d7cd011db47",
    "https://ml.azure.com/experiments/id/cbd45cd3-4fd8-4922-b82a-124527cc98ee/runs/cool_cheese_f8v978x9n0?wsid=/subscriptions/acc09744-1ee3-4242-b375-93421c63af0c/resourceGroups/PPML/providers/Microsoft.MachineLearningServices/workspaces/M365Research-PPML-EUS&tid=72f988bf-86f1-41af-91ab-2d7cd011db47",
    # prefix_length = 20
    "https://ml.azure.com/experiments/id/cbd45cd3-4fd8-4922-b82a-124527cc98ee/runs/maroon_hominy_h1pqfbxdqt?wsid=/subscriptions/acc09744-1ee3-4242-b375-93421c63af0c/resourceGroups/PPML/providers/Microsoft.MachineLearningServices/workspaces/M365Research-PPML-EUS&tid=72f988bf-86f1-41af-91ab-2d7cd011db47",
    "https://ml.azure.com/experiments/id/cbd45cd3-4fd8-4922-b82a-124527cc98ee/runs/salmon_sprout_5ycm2860jq?wsid=/subscriptions/acc09744-1ee3-4242-b375-93421c63af0c/resourceGroups/PPML/providers/Microsoft.MachineLearningServices/workspaces/M365Research-PPML-EUS&tid=72f988bf-86f1-41af-91ab-2d7cd011db47",
    "https://ml.azure.com/experiments/id/cbd45cd3-4fd8-4922-b82a-124527cc98ee/runs/frank_okra_2nhggxmsd2?wsid=/subscriptions/acc09744-1ee3-4242-b375-93421c63af0c/resourceGroups/PPML/providers/Microsoft.MachineLearningServices/workspaces/M365Research-PPML-EUS&tid=72f988bf-86f1-41af-91ab-2d7cd011db47",
    "https://ml.azure.com/experiments/id/cbd45cd3-4fd8-4922-b82a-124527cc98ee/runs/dreamy_napkin_5fh4v95dky?wsid=/subscriptions/acc09744-1ee3-4242-b375-93421c63af0c/resourceGroups/PPML/providers/Microsoft.MachineLearningServices/workspaces/M365Research-PPML-EUS&tid=72f988bf-86f1-41af-91ab-2d7cd011db47",
    # prefix_length = 30
    "https://ml.azure.com/experiments/id/cbd45cd3-4fd8-4922-b82a-124527cc98ee/runs/yellow_van_0d58vtdw1t?wsid=/subscriptions/acc09744-1ee3-4242-b375-93421c63af0c/resourceGroups/PPML/providers/Microsoft.MachineLearningServices/workspaces/M365Research-PPML-EUS&tid=72f988bf-86f1-41af-91ab-2d7cd011db47",
    "https://ml.azure.com/experiments/id/cbd45cd3-4fd8-4922-b82a-124527cc98ee/runs/keen_book_lgsjnr0wtf?wsid=/subscriptions/acc09744-1ee3-4242-b375-93421c63af0c/resourceGroups/PPML/providers/Microsoft.MachineLearningServices/workspaces/M365Research-PPML-EUS&tid=72f988bf-86f1-41af-91ab-2d7cd011db47",
    "https://ml.azure.com/experiments/id/cbd45cd3-4fd8-4922-b82a-124527cc98ee/runs/keen_kitchen_rybbjp9r7w?wsid=/subscriptions/acc09744-1ee3-4242-b375-93421c63af0c/resourceGroups/PPML/providers/Microsoft.MachineLearningServices/workspaces/M365Research-PPML-EUS&tid=72f988bf-86f1-41af-91ab-2d7cd011db47",
])


In [ ]:
data = pd.DataFrame([{k.replace("AZUREML_PARAMETER_", ""): v for k, v in job.get_node("get_ood_canaries").details["runDefinition"]["environmentVariables"].items()} for job in jobs])
data["prefix_length"] = data["prefix_length"].fillna(0).astype(int)  # prefix_length 0 experiments don't have that parameter so set it to 0 if it's missing

In [ ]:
data["min_ppl"] = data["min_ppl"].astype(float)
data["max_ppl"] = data["max_ppl"].astype(float)
data["prefix_length"] = data["prefix_length"].astype(int)
data["ppl"] = (data["min_ppl"] + data["max_ppl"]) / 2
data

In [ ]:
def load_input(job: Job, name: str):
    with TemporaryDirectory() as tmpdir:
        job.download_input(name, path=tmpdir)
        return load_from_disk(tmpdir)


In [ ]:
def compute_metrics(job):
    estimate_privacy = job.get_node("estimate_privacy")
    scores = load_input(estimate_privacy, "scores")
    challenge_bits = load_input(estimate_privacy, "challenge_bits")

    fpr, tpr, _ = roc_curve(challenge_bits["challenge_bit"], scores["score"])
    metrics = {"FPR": fpr, "TPR": tpr}
    metrics["AuC"] = auc(fpr, tpr)
    for target_fpr in [0.01, 0.05, 0.1]:
        metrics[f"TPR@FPR={target_fpr}"] = np.interp(target_fpr, fpr, tpr)
        
    return metrics

In [ ]:
data = pd.concat([data, pd.DataFrame([compute_metrics(job) for job in jobs])], axis=1)

In [ ]:
data = data.sort_values("ppl")

In [ ]:
plt.figure(figsize=(6,6))
plot_options = {
    0: {"color": "darkred", "label": "In-distribution prefix length: 0"},
    10: {"color": "darkblue", "label": "In-distribution prefix length: 10"},
    20: {"color": "darkgreen", "label": "In-distribution prefix length: 20"},
    30: {"color": "darkorange", "label": "In-distribution prefix length: 30"},
}
for prefix in plot_options:
    ppl = data[data["prefix_length"] == prefix]["ppl"]
    auc = data[data["prefix_length"] == prefix]["AuC"]
    plt.plot(ppl, auc, "-o", **plot_options[prefix])

plt.axhline(y=0.5, color='black', linestyle='--', alpha=0.5, label = 'Random guess baseline')


plt.xticks([10**k for k in (0, 1, 2, 3, 4, 5)])
plt.yticks([0.5, 0.6, 0.7, 0.8, 0.9, 1.0], labels=['0.5', '0.6', '0.7', '0.8', '0.9', '1.0'])

# Enable the grid
plt.grid(True, which="major", ls="--", alpha=0.8)

plt.legend(loc='upper right', fontsize=14)
plt.xlabel('Canary perplexity', fontsize=16)
plt.ylabel('AUC', fontsize=16)
plt.ylim(0.4, 1.02)
plt.xscale('log')
plt.show()

In [ ]:
df_auc = {
    "prefix_0": pd.DataFrame({'ppl': data[data['prefix_length']==0]['ppl'].values, 'auc': data[data['prefix_length']==0]['AuC'].values}),
    "prefix_10": pd.DataFrame({'ppl': data[data['prefix_length']==10]['ppl'].values, 'auc': data[data['prefix_length']==10]['AuC'].values}),
    "prefix_20": pd.DataFrame({'ppl': data[data['prefix_length']==20]['ppl'].values, 'auc': data[data['prefix_length']==20]['AuC'].values}),
    "prefix_30": pd.DataFrame({'ppl': data[data['prefix_length']==30]['ppl'].values, 'auc': data[data['prefix_length']==30]['AuC'].values}),
}
df_auc

In [ ]:
perp = 31
fig, ax = plt.subplots(1, 2, figsize=(12, 6))
plot_options = {
    0: {"color": "darkred", "label": "In-distribution prefix length: 0", "linestyle": "-"},
    10: {"color": "darkblue", "label": "In-distribution prefix length: 10", "linestyle": "--"},
    20: {"color": "darkgreen", "label": "In-distribution prefix length: 20", "linestyle": "-."},
    30: {"color": "darkorange", "label": "In-distribution prefix length: 30", "linestyle": ":"},
}

for prefix in plot_options:
    data_i = data[(perp*0.9 <= data["ppl"]) & (data["ppl"] <= perp*1.1)]
    data_i = data_i[data_i["prefix_length"] == prefix]
    fpr = data_i["FPR"].values[0]
    tpr = data_i["TPR"].values[0]
    ax[0].plot(fpr, tpr, **plot_options[prefix])
    ax[1].plot(fpr, tpr, **plot_options[prefix])


ax[0].plot([0, 1], [0, 1], "--", color="black", alpha=0.5, label="Random guess baseline")
ax[1].plot([0, 1], [0, 1], "--", color="black", alpha=0.5, label="Random guess baseline")
ax[0].set_xlabel("False positive rate", fontsize=16)
ax[0].set_ylabel("True positive rate", fontsize=16)
ax[1].set_xlabel("False positive rate", fontsize=16)
ax[1].set_ylabel("True positive rate", fontsize=16)
ax[1].legend(loc="lower right", fontsize=14)
ax[0].grid(True, which="major", ls="--", alpha=0.8)
ax[1].grid(True, which="major", ls="--", alpha=0.8)
ax[1].set_xscale("log")
ax[1].set_yscale("log")
ax[0].set_xlim(0, 1)
ax[0].set_ylim(0, 1)


In [ ]:
data_ppl = data[(perp*0.9 <= data["ppl"]) & (data["ppl"] <= perp*1.1)]
df_roc = {
    "prefix_0": pd.DataFrame({'fpr': data_ppl[data_ppl['prefix_length']==0]['FPR'].values[0], 'tpr': data_ppl[data_ppl['prefix_length']==0]['TPR'].values[0]}),
    "prefix_10": pd.DataFrame({'fpr': data_ppl[data_ppl['prefix_length']==10]['FPR'].values[0], 'tpr': data_ppl[data_ppl['prefix_length']==10]['TPR'].values[0]}),
    "prefix_20": pd.DataFrame({'fpr': data_ppl[data_ppl['prefix_length']==20]['FPR'].values[0], 'tpr': data_ppl[data_ppl['prefix_length']==20]['TPR'].values[0]}),
    "prefix_30": pd.DataFrame({'fpr': data_ppl[data_ppl['prefix_length']==30]['FPR'].values[0], 'tpr': data_ppl[data_ppl['prefix_length']==30]['TPR'].values[0]}),
}

In [ ]:
load_dotenv()
overleaf = Project(path=os.getenv("LATEX_GIT_PATH"))
for name, df in df_auc.items():
    overleaf.add_dataframe_as_tsv(df, f"data/prefix/sst2/auc/{name}.tsv")
for name, df in df_roc.items():
    overleaf.add_dataframe_as_tsv(df, f"data/prefix/sst2/roc/{name}.tsv")

In [ ]:
result_rows = []
for prefix, roc_data in df_roc.items():
    # Sort the FPR and TPR arrays if not sorted
    sorted_indices = np.argsort(roc_data['fpr'])
    fpr = np.array(roc_data['fpr'])[sorted_indices]
    tpr = np.array(roc_data['tpr'])[sorted_indices]
    
    tpr_0_01 = np.interp(0.01, fpr, tpr)
    tpr_0_05 = np.interp(0.05, fpr, tpr)
    tpr_0_1 = np.interp(0.1, fpr, tpr)
    
    result_rows.append({
        'prefix': prefix,
        'TPR@FPR=0.01': tpr_0_01,
        'TPR@FPR=0.05': tpr_0_05,
        'TPR@FPR=0.1': tpr_0_1,
    })

df_result = pd.DataFrame(result_rows)
print(df_result)